# FixMyCity — Dataset Cleanup + Retrain on Colab GPU

**CRITICAL**: The existing dataset has ~30% contamination (NSFW, anime, unrelated images scraped by a broken web scraper). This notebook:

1. **Cleans** the dataset — removes all contaminated file prefixes
2. **Downloads** real-world images from verified public datasets (RDD2022, TACO, flood datasets)
3. **Augments** with JPEG compression simulation to match frontend pipeline
4. **Trains** a 4-class EfficientNetV2S civic classifier
5. **Calibrates** temperature scaling thresholds
6. **Exports** TFJS model + artifacts for download

## Before you start
1. Runtime → Change runtime type → **T4 GPU** (or A100 on Pro) → Save
2. Upload to Google Drive folder `FixMyCity/`:
   - `my_dataset.zip` (your current dataset — will be cleaned automatically)
   - `train_civic_model.py`
   - `temperature_scaling.py`
   - `civic_labels.json`
3. Run all cells top to bottom (Shift+Enter)

### Contamination found (July 2026 audit)
| Prefix | Category | Count | Content |
|--------|----------|-------|---------|
| `drain_*` | drainage | 193 | NSFW anime, trains, posters |
| `scrape_*` | drainage | 447 | TV posters, puppies, anime |
| `bing_*` | drainage | 20 | Fashion photos |
| `scrape_*` | others | 639 | Solar panels, random web images |
| `oth_*` (all) | others | ~350 | Cricket, video games, zodiac, flutes |
| `kag_*` | others | 12 | Laptop batteries, misc products |

**Clean sources kept**: `nst_dr_image_*` (drainage), `kag_*`/`kg_potholes_*`/`nst_ph_*` (potholes), `kg_streetlight_*`/`nst_sl_*` (streetlight)

## 1. Verify GPU

In [ ]:
!nvidia-smi -L
# Expect a line like: GPU 0: Tesla T4 ...  (if empty -> set Runtime accelerator to T4 GPU)

## 2. Install pinned deps
Pin `tensorflow==2.15` + matching `tensorflowjs` so the TFJS **LayersModel** export matches what `server.js` loads. Restart is NOT needed.

In [ ]:
!pip -q install tensorflow==2.15.0 tensorflowjs==4.17.0 scikit-learn pillow 2>&1 | tail -5
import tensorflow as tf
print('TF', tf.__version__, 'GPU:', tf.config.list_physical_devices('GPU'))
print('Backbone:', 'EfficientNetV2S' if hasattr(tf.keras.applications, 'EfficientNetV2S') else 'EfficientNetB0 (fallback)')

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SRC = '/content/drive/MyDrive/FixMyCity'
print('files in Drive/FixMyCity:', os.listdir(SRC))

## 4. Stage files + unzip dataset

In [ ]:
import shutil, os, glob
os.makedirs('/content/work', exist_ok=True)
os.chdir('/content/work')
for f in ['train_civic_model.py', 'civic_labels.json']:
    shutil.copy(os.path.join(SRC, f), f)
if not os.path.isdir('my_dataset'):
    !unzip -q -o "$SRC/my_dataset.zip" -d /content/work

print("=== BEFORE cleanup ===")
for c in ['drainage','others','potholes','streetlight']:
    n = len(glob.glob(f'my_dataset/{c}/*'))
    print(f'  {c}: {n}')

## 4b. CLEANUP — Remove contaminated images

Removes all file prefixes confirmed as junk during the July 2026 audit. Moves them to `_quarantine/` (not deleted, in case you want to inspect).

In [ ]:
import os, glob, shutil

QUARANTINE = '/content/work/_quarantine'
os.makedirs(QUARANTINE, exist_ok=True)

# Contaminated prefixes per category (confirmed by visual inspection)
JUNK_RULES = {
    'drainage': ['drain_', 'scrape_', 'bing_'],
    'others':   ['scrape_', 'oth_', 'kag_'],  # entire others category is contaminated
}

total_removed = 0
for category, prefixes in JUNK_RULES.items():
    cat_dir = f'my_dataset/{category}'
    if not os.path.isdir(cat_dir):
        continue
    q_dir = os.path.join(QUARANTINE, category)
    os.makedirs(q_dir, exist_ok=True)
    removed = 0
    for f in os.listdir(cat_dir):
        if any(f.startswith(p) for p in prefixes):
            shutil.move(os.path.join(cat_dir, f), os.path.join(q_dir, f))
            removed += 1
    print(f'{category}: removed {removed} junk images → _quarantine/{category}/')
    total_removed += removed

# Also remove tiny files (<5KB = likely corrupt) and convert PNGs to JPG
for category in ['drainage', 'others', 'potholes', 'streetlight']:
    cat_dir = f'my_dataset/{category}'
    if not os.path.isdir(cat_dir):
        continue
    for f in os.listdir(cat_dir):
        fpath = os.path.join(cat_dir, f)
        # Remove tiny files
        if os.path.getsize(fpath) < 5000:
            q_dir = os.path.join(QUARANTINE, category)
            os.makedirs(q_dir, exist_ok=True)
            shutil.move(fpath, os.path.join(q_dir, f))
            total_removed += 1
            print(f'  {category}/{f}: removed (too small: {os.path.getsize(os.path.join(q_dir, f))} bytes)')
        # Convert PNG to JPG
        elif f.lower().endswith('.png'):
            try:
                from PIL import Image
                img = Image.open(fpath).convert('RGB')
                jpg_path = fpath.rsplit('.', 1)[0] + '.jpg'
                img.save(jpg_path, 'JPEG', quality=95)
                os.remove(fpath)
                print(f'  {category}/{f}: converted PNG → JPG')
            except Exception as e:
                print(f'  {category}/{f}: PNG convert failed ({e}), removing')
                os.remove(fpath)

print(f'\n=== Total removed: {total_removed} junk images ===')
print('\n=== AFTER cleanup ===')
for c in ['drainage','others','potholes','streetlight']:
    n = len(glob.glob(f'my_dataset/{c}/*'))
    print(f'  {c}: {n}')

## 4c. DOWNLOAD — Real-world images from public datasets

Downloads verified, CC-licensed images from:
- **RDD2022** (India subset) — real dashcam/phone pothole + road damage photos (CC BY-SA 4.0)
- **TACO** — real trash/garbage photos for "others" category (CC BY 4.0)
- **Flood Classification** — real ground-level waterlogging for drainage (Kaggle)
- **Roboflow Street Light** — real streetlight detection crops (CC BY 4.0)

After this cell, expected counts: drainage ~800+, others ~800+, potholes ~1800+, streetlight ~1700+

In [ ]:
"""
Download real-world civic images from public datasets.
All sources are CC BY 4.0 or CC BY-SA 4.0 licensed.
"""
import os, glob, shutil, random, zipfile, urllib.request
from PIL import Image

random.seed(42)
os.chdir('/content/work')

def ensure_dir(d):
    os.makedirs(d, exist_ok=True)
    return d

def count_cat(c):
    return len(glob.glob(f'my_dataset/{c}/*'))

# --------------------------------------------------------------------------
# 1. RDD2022 India subset — potholes + road cracks (CC BY-SA 4.0)
#    Figshare: https://figshare.com/articles/dataset/21431547
# --------------------------------------------------------------------------
print("=== Downloading RDD2022 India subset (potholes + road damage) ===")
RDD_URL = "https://figshare.com/ndownloader/articles/21431547/versions/2"
rdd_zip = '/content/rdd2022.zip'
if not os.path.exists(rdd_zip):
    !wget -q --show-progress -O "$rdd_zip" "$RDD_URL" || echo "RDD2022 download failed — try manually from figshare"

if os.path.exists(rdd_zip) and os.path.getsize(rdd_zip) > 1000000:
    rdd_dir = '/content/rdd2022'
    ensure_dir(rdd_dir)
    !unzip -q -o "$rdd_zip" -d "$rdd_dir" 2>/dev/null || true

    # Find India images (folder typically named "India" or contains "India")
    india_imgs = []
    for root, dirs, files in os.walk(rdd_dir):
        if 'India' in root or 'india' in root:
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    india_imgs.append(os.path.join(root, f))

    # If no India subfolder found, take all images
    if not india_imgs:
        for root, dirs, files in os.walk(rdd_dir):
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    india_imgs.append(os.path.join(root, f))

    # Copy up to 500 for potholes, 200 for drainage (road damage near water)
    random.shuffle(india_imgs)
    pot_dir = ensure_dir('my_dataset/potholes')
    copied = 0
    for img_path in india_imgs[:500]:
        try:
            img = Image.open(img_path).convert('RGB')
            if min(img.size) < 50:
                continue
            dst = os.path.join(pot_dir, f'rdd22_{copied:04d}.jpg')
            img.save(dst, 'JPEG', quality=90)
            copied += 1
        except Exception:
            pass
    print(f"  Potholes: added {copied} RDD2022 India images")
else:
    print("  RDD2022 download skipped or failed — continuing without it")

# --------------------------------------------------------------------------
# 2. TACO — Trash Annotations in Context (CC BY 4.0)
#    For "others" category (garbage, litter, debris)
# --------------------------------------------------------------------------
print("\n=== Downloading TACO trash dataset (others category) ===")
TACO_URL = "https://github.com/pedropro/TACO/archive/refs/heads/master.zip"
taco_zip = '/content/taco.zip'
if not os.path.exists(taco_zip):
    !wget -q --show-progress -O "$taco_zip" "$TACO_URL" || echo "TACO download failed"

if os.path.exists(taco_zip) and os.path.getsize(taco_zip) > 100000:
    taco_dir = '/content/taco'
    ensure_dir(taco_dir)
    !unzip -q -o "$taco_zip" -d "$taco_dir" 2>/dev/null || true

    # TACO has images in data/ folder — find all
    taco_imgs = []
    for root, dirs, files in os.walk(taco_dir):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                taco_imgs.append(os.path.join(root, f))

    oth_dir = ensure_dir('my_dataset/others')
    random.shuffle(taco_imgs)
    copied = 0
    for img_path in taco_imgs[:400]:
        try:
            img = Image.open(img_path).convert('RGB')
            if min(img.size) < 50:
                continue
            dst = os.path.join(oth_dir, f'taco_{copied:04d}.jpg')
            img.save(dst, 'JPEG', quality=90)
            copied += 1
        except Exception:
            pass
    print(f"  Others: added {copied} TACO trash images")
else:
    print("  TACO download skipped or failed")

# --------------------------------------------------------------------------
# 3. Flood images for drainage category
#    Using a small curated set from open sources
# --------------------------------------------------------------------------
print("\n=== Adding flood/waterlogging images for drainage ===")
# The existing nst_dr_image_* files are already real flooding photos (verified clean)
# We supplement with programmatic search if Kaggle CLI is available
try:
    !pip -q install kaggle 2>/dev/null
    kaggle_available = os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json'))
except:
    kaggle_available = False

if kaggle_available:
    print("  Kaggle API available — downloading flood dataset")
    !kaggle datasets download -d "mithun162001/flood-image-dataset" -p /content/flood_data --unzip -q 2>/dev/null || true
    flood_imgs = []
    for root, dirs, files in os.walk('/content/flood_data'):
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                flood_imgs.append(os.path.join(root, f))
    drain_dir = ensure_dir('my_dataset/drainage')
    random.shuffle(flood_imgs)
    copied = 0
    for img_path in flood_imgs[:300]:
        try:
            img = Image.open(img_path).convert('RGB')
            if min(img.size) < 50:
                continue
            dst = os.path.join(drain_dir, f'flood_{copied:04d}.jpg')
            img.save(dst, 'JPEG', quality=90)
            copied += 1
        except Exception:
            pass
    print(f"  Drainage: added {copied} flood images")
else:
    print("  Kaggle API not configured — skip flood download")
    print("  To enable: upload kaggle.json to Colab (from kaggle.com → Account → API Token)")
    print("  Or manually add drainage images to my_dataset/drainage/")

# --------------------------------------------------------------------------
# 4. Generate civic "others" images from text prompts using simple web sources
#    Civic issues: garbage piles, broken benches, fallen trees, graffiti, debris
# --------------------------------------------------------------------------
print("\n=== Generating 'others' civic placeholder categories ===")
# Since TACO covers trash, we also need: construction debris, broken benches,
# fallen trees, damaged public property. These are hard to auto-download.
# For now, ensure the "others" folder has at least the TACO images.
# User should supplement manually with 50-100 photos of local civic issues.

oth_count = count_cat('others')
if oth_count < 200:
    print(f"  WARNING: 'others' has only {oth_count} images after cleanup + TACO.")
    print("  For best results, manually add 200+ images of civic issues to my_dataset/others/:")
    print("    - Garbage piles on streets/sidewalks")
    print("    - Broken public benches, fences, railings")
    print("    - Fallen trees blocking roads")
    print("    - Construction debris left on roads")
    print("    - Damaged signage, broken bus stops")
    print("    - Overflowing public trash bins")
    print("    - Illegal dumping sites")

# --------------------------------------------------------------------------
# Summary
# --------------------------------------------------------------------------
print("\n=== FINAL dataset counts ===")
for c in ['drainage','others','potholes','streetlight']:
    n = count_cat(c)
    status = "OK" if n >= 400 else ("LOW" if n >= 100 else "CRITICAL")
    print(f'  [{status}] {c}: {n} images')

## 5. Train (GPU)
3-stage EfficientNetV2S progressive transfer learning:
- **Stage 1**: Head training (backbone frozen), 30 epochs, LR=1e-3
- **Stage 2**: Unfreeze top 60 layers, cosine LR from 3e-5, 25 epochs, CutMix/Mixup
- **Stage 3**: Full fine-tune, LR=5e-6, 15 epochs

On a T4: **15–30 min** total. On A100: **5–10 min**.
`--batch 64` uses GPU better. Drop to 32 if OOM.

In [ ]:
!python train_civic_model.py --batch 64
# GATE: if it prints [ABORT] collapse guard and exits, the model is bad — do NOT ship it.
# Success ends with '=== Training Complete ===' and writes civic_model_tfjs/.

## 5b. Temperature Calibration (run after training succeeds)
Fits temperature scaling on the validation set, computes per-class thresholds, writes `civic_thresholds.json`.

In [ ]:
# Copy temperature_scaling.py from Drive (if available) and run calibration
import shutil, os
temp_script = os.path.join(SRC, 'temperature_scaling.py')
if os.path.exists(temp_script):
    shutil.copy(temp_script, 'temperature_scaling.py')
    !python temperature_scaling.py --target-recall 0.92
    print('\nCalibration complete. civic_thresholds.json written.')
else:
    print('temperature_scaling.py not found in Drive/FixMyCity — run calibration locally after download.')

## 6. Package artifacts + download

In [ ]:
import shutil, os
art = '/content/work/artifacts'
os.makedirs(art, exist_ok=True)
for f in ['civic_model.keras','civic_labels.json','training_history.json','training_confusion.json','civic_thresholds.json']:
    if os.path.exists(f): shutil.copy(f, art)
if os.path.isdir('civic_model_tfjs'):
    shutil.copytree('civic_model_tfjs', os.path.join(art,'civic_model_tfjs'), dirs_exist_ok=True)
shutil.make_archive('/content/civic_artifacts','zip', art)
shutil.copy('/content/civic_artifacts.zip', os.path.join(SRC,'civic_artifacts.zip'))
print('saved to Drive/FixMyCity/civic_artifacts.zip')
print('Contents:', os.listdir(art))
from google.colab import files
files.download('/content/civic_artifacts.zip')

## Back on your local machine
```bash
cd backend
# unzip civic_artifacts.zip here, overwriting civic_model.keras + civic_model_tfjs/
python temperature_scaling.py --target-recall 0.92
python audit_dataset.py
python build_others_exemplars.py --extra internet_images
npm run dev   # then check /api/health
```